# Baseline Anomaly Detection Policies

Evaluates three baseline policies on per-instance metrics:
- **Do Nothing**: Always predicts False (no anomaly)
- **Do All**: Always predicts True (anomaly)
- **Flip Coin**: Random 50/50 prediction

In [1]:
import json
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

np.random.seed(42)

## Load Data

In [2]:
# Load ground truth
with open('ground_truth.json', 'r') as f:
    ground_truth = json.load(f)

# Load instances
instances = {}
for f in Path('instances').glob('*.json'):
    with open(f) as file:
        data = json.load(file)
        instances[data['id']] = data

# Load sensor data
sensor_data = {}
for f in Path('sensor_data').glob('*.json'):
    with open(f) as file:
        sensor_data[f.stem] = json.load(file)

print(f"Loaded {len(instances)} instances with {len(sensor_data)} sensor data files")
print(f"Ground truth contains {len(ground_truth)} anomaly periods")

Loaded 1 instances with 1 sensor data files
Ground truth contains 1 anomaly periods


## Create Ground Truth Labels

In [3]:
def create_labels(sensor_data, ground_truth):
    """Create ground truth labels for each timestamp."""
    # Map instance_id to anomaly periods
    anomaly_periods = {}
    for entry in ground_truth:
        instance_id = entry['id']
        start = datetime.fromisoformat(entry['start_datetime_utc'].replace('Z', '+00:00'))
        end = datetime.fromisoformat(entry['end_datetime_utc'].replace('Z', '+00:00'))
        if instance_id not in anomaly_periods:
            anomaly_periods[instance_id] = []
        anomaly_periods[instance_id].append((start, end))
    
    # Create labels
    labels = {}
    for instance_id, readings in sensor_data.items():
        labels[instance_id] = {}
        for timestamp_str in readings.keys():
            timestamp = datetime.fromisoformat(timestamp_str.replace('Z', '+00:00'))
            is_anomaly = False
            if instance_id in anomaly_periods:
                for start, end in anomaly_periods[instance_id]:
                    if start <= timestamp <= end:
                        is_anomaly = True
                        break
            labels[instance_id][timestamp_str] = is_anomaly
    return labels

gt_labels = create_labels(sensor_data, ground_truth)
total_timestamps = sum(len(labels) for labels in gt_labels.values())
total_anomalies = sum(sum(labels.values()) for labels in gt_labels.values())
print(f"Total timestamps: {total_timestamps}, Anomalies: {total_anomalies} ({total_anomalies/total_timestamps*100:.2f}%)")

Total timestamps: 61, Anomalies: 14 (22.95%)


## Baseline Policies

In [4]:
def do_nothing_policy(sensor_data):
    """Always predict False."""
    return {iid: {ts: False for ts in readings.keys()} 
            for iid, readings in sensor_data.items()}

def do_all_policy(sensor_data):
    """Always predict True."""
    return {iid: {ts: True for ts in readings.keys()} 
            for iid, readings in sensor_data.items()}

def flip_coin_policy(sensor_data, seed=42):
    """Random 50/50 prediction."""
    np.random.seed(seed)
    return {iid: {ts: bool(np.random.rand() > 0.5) for ts in readings.keys()} 
            for iid, readings in sensor_data.items()}

# Generate predictions
predictions = {
    'Do Nothing': do_nothing_policy(sensor_data),
    'Do All': do_all_policy(sensor_data),
    'Flip Coin': flip_coin_policy(sensor_data)
}

print("Generated predictions for all policies")

Generated predictions for all policies


## Per-Instance Evaluation

In [5]:
def evaluate_per_instance(gt_labels, predictions, policy_name):
    """Evaluate predictions per instance."""
    results = []
    for instance_id in gt_labels.keys():
        if instance_id not in predictions:
            continue
        
        y_true = [gt_labels[instance_id][ts] for ts in gt_labels[instance_id].keys() 
                  if ts in predictions[instance_id]]
        y_pred = [predictions[instance_id][ts] for ts in gt_labels[instance_id].keys() 
                  if ts in predictions[instance_id]]
        
        if len(y_true) == 0:
            continue
        
        results.append({
            'Policy': policy_name,
            'Instance': instance_id[:8],
            'Rule': instances.get(instance_id, {}).get('rule', 'Unknown'),
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred, zero_division=0)
        })
    return pd.DataFrame(results)

# Evaluate all policies
all_results = pd.concat([
    evaluate_per_instance(gt_labels, preds, name) 
    for name, preds in predictions.items()
], ignore_index=True)

print(f"Evaluated {len(all_results)} policy-instance combinations")

Evaluated 3 policy-instance combinations


## Results Summary

In [6]:
# Summary statistics by policy
summary = all_results.groupby('Policy')[['Accuracy', 'Precision', 'Recall', 'F1']].agg(['mean', 'sem'])
print("\n=== Summary Statistics by Policy ===")
print(summary.round(4))


=== Summary Statistics by Policy ===
           Accuracy     Precision     Recall          F1    
               mean sem      mean sem   mean sem    mean sem
Policy                                                      
Do All       0.2295 NaN    0.2295 NaN    1.0 NaN  0.3733 NaN
Do Nothing   0.7705 NaN    0.0000 NaN    0.0 NaN  0.0000 NaN
Flip Coin    0.5410 NaN    0.2500 NaN    0.5 NaN  0.3333 NaN
